# Impact of 6G Network Performance on Manufacturing Efficiency

This Colab notebook performs the full EDA and statistical diagnostics for the smart-factory project.

In [ ]:
!pip -q install pandas numpy scipy matplotlib seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats


In [ ]:
# Preferred: load directly from GitHub after replacing these two values.
GITHUB_USERNAME = 'YOUR_GITHUB_USERNAME'
REPOSITORY_NAME = 'YOUR_REPOSITORY_NAME'
RAW_URL = f'https://raw.githubusercontent.com/{GITHUB_USERNAME}/{REPOSITORY_NAME}/main/data/Thales_Group_Manufacturing.csv'

if GITHUB_USERNAME != 'YOUR_GITHUB_USERNAME':
    df = pd.read_csv(RAW_URL)
else:
    # First-run fallback: upload the CSV manually.
    from google.colab import files
    uploaded = files.upload()
    file_name = next(iter(uploaded))
    df = pd.read_csv(file_name)

print(df.shape)
df.head()


## 1. Data quality and descriptive statistics

In [ ]:
print(df.info())
print('\nMissing values:')
print(df.isna().sum())
print('\nDescriptive statistics:')
display(df.describe(include='all').T)

## 2. Network performance profiling

In [ ]:
df['Latency_Band'] = pd.qcut(df['Network_Latency_ms'], 3, labels=['Low','Medium','High'], duplicates='drop')
df['Packet_Loss_Band'] = pd.qcut(df['Packet_Loss_%'], 3, labels=['Low','Medium','High'], duplicates='drop')
df['Network_Quality'] = np.select([
    (df.Latency_Band=='Low') & (df.Packet_Loss_Band=='Low'),
    (df.Latency_Band=='High') | (df.Packet_Loss_Band=='High')],
    ['High','Low'], default='Medium')
print(df['Network_Quality'].value_counts())
fig, ax = plt.subplots(figsize=(8,5)); ax.hist(df['Network_Latency_ms'], bins=30); ax.set_title('Network Latency Distribution'); ax.set_xlabel('Latency (ms)'); ax.set_ylabel('Frequency'); plt.show()
fig, ax = plt.subplots(figsize=(8,5)); ax.hist(df['Packet_Loss_%'], bins=30); ax.set_title('Packet Loss Distribution'); ax.set_xlabel('Packet Loss (%)'); ax.set_ylabel('Frequency'); plt.show()

## 3. Network vs efficiency

In [ ]:
display(pd.crosstab(df['Network_Quality'], df['Efficiency_Status'], normalize='index').round(3))
table=pd.crosstab(df['Network_Quality'], df['Efficiency_Status'])
chi2,p,dof,expected=stats.chi2_contingency(table)
cramers_v=np.sqrt((chi2/table.values.sum())/min(table.shape[0]-1, table.shape[1]-1))
print(f'Chi-square={chi2:.4f}, p={p:.6g}, Cramers V={cramers_v:.4f}')

In [ ]:
sample=df.sample(min(8000,len(df)), random_state=42)
fig, ax = plt.subplots(figsize=(8,5));
for label, g in sample.groupby('Efficiency_Status'):
    ax.scatter(g['Network_Latency_ms'], g['Production_Speed_units_per_hr'], s=8, alpha=.35, label=label)
ax.set_title('Latency vs Production Speed'); ax.set_xlabel('Latency (ms)'); ax.set_ylabel('Units/hour'); ax.legend(); plt.show()
slope,intercept,r,p,se=stats.linregress(df['Network_Latency_ms'],df['Production_Speed_units_per_hr'])
print(f'Latency sensitivity slope = {slope:.4f} units/hour per ms; R²={r*r:.6f}; p={p:.6g}')

## 4. Packet loss diagnostics

In [ ]:
fig, ax = plt.subplots(figsize=(8,5)); ax.scatter(sample['Packet_Loss_%'], sample['Error_Rate_%'], s=8, alpha=.35); ax.set_title('Packet Loss vs Error Rate'); ax.set_xlabel('Packet Loss (%)'); ax.set_ylabel('Error Rate (%)'); plt.show()
fig, ax = plt.subplots(figsize=(8,5)); ax.scatter(sample['Packet_Loss_%'], sample['Quality_Control_Defect_Rate_%'], s=8, alpha=.35); ax.set_title('Packet Loss vs Defect Rate'); ax.set_xlabel('Packet Loss (%)'); ax.set_ylabel('Defect Rate (%)'); plt.show()
low=df.loc[df.Packet_Loss_Band=='Low','Production_Speed_units_per_hr'].mean(); high=df.loc[df.Packet_Loss_Band=='High','Production_Speed_units_per_hr'].mean(); print(f'Packet Loss Impact Ratio={(low-high)/low:.6f}')

## 5. Correlation matrix and operation-mode analysis

In [ ]:
cols=['Network_Latency_ms','Packet_Loss_%','Production_Speed_units_per_hr','Error_Rate_%','Quality_Control_Defect_Rate_%']
display(df[cols].corr().round(4))
display(df.groupby('Operation_Mode')[cols].mean().round(3))

## 6. Research interpretation checklist

- Use effect size, not p-value alone.
- A statistically significant result with a near-zero correlation may have little practical importance in a 100,000-row dataset.
- Compare Active/Idle/Maintenance modes before claiming a network effect.
- Treat quartiles as dataset-specific monitoring benchmarks, not universal 6G engineering limits.
- Do not claim causation without controlled experiments or stronger causal design.